# Reference implementation — INjecting BM25 Scores


It measures **injected vs non-injected**. Comparison against the paper's published figures happens separately.

Five steps: **retrieve → format the score → build the input → score → evaluate.**

##  Configuration

Every constant is quoted from the paper. The two that matter most: GLOBAL_MAX_BM25 = 50

In [1]:
from pathlib import Path

GLOBAL_MIN_BM25 = 0
GLOBAL_MAX_BM25 = 50

# BM25 98 is is the max score

INJECT_MAX_INT = 196

RERANK_DEPTH       = 1000 
QUERY_MAX_TOKENS   = 30
PASSAGE_MAX_TOKENS = 200 
BM25_K1, BM25_B    = 0.82, 0.68 # "tuned parameters from the Anserini documentation"

COLLECTIONS = {
    "TREC DL'19":  {"irds": "msmarco-passage/trec-dl-2019/judged", "topics": 43, "graded": True},
    "TREC DL'20":  {"irds": "msmarco-passage/trec-dl-2020/judged", "topics": 54, "graded": True},
    "MSMARCO DEV": {"irds": "msmarco-passage/dev/small",           "topics": 6980, "graded": False},
}


# ── our checkpoints ───────────────────────────────────────────────────────────────────
# Trained under the paper cross-entropy loss, early stopping on DL'20 nDCG@10,
# Eq.3 ordering (score BETWEEN query and passage). Identical data, seed and schedule;
# the two differ only in whether the score is injected.
MODELS = {
    "CAT":     {"repo": "Amdestya/ce-cat-minilm-l12",     "inject": None},
    "BM25CAT": {"repo": "Amdestya/ce-bm25cat-minilm-l12", "inject": "score_middle"},
}

DATA = Path("./paper_tables_data"); DATA.mkdir(exist_ok=True)
OUT  = Path("./paper_tables_out");  OUT.mkdir(exist_ok=True)


## 1a · Setup

Cache paths, imports, and which collection to run. The index is ~3.9 GB and the HuggingFace cache another ~3.2 GB, so both are redirected out of your home directory.

In [2]:
"""Step 1a — imports, and the collection to run."""
import os, json, urllib.request
import pandas as pd

# These must be set BEFORE pyterrier is imported -- it reads PYTERRIER_HOME at import time.
import pyterrier as pt

COLLECTION = "TREC DL'19"      # one at a time. DEV is 6,980 queries -> ~2 h per model.
cfg = COLLECTIONS[COLLECTION]

print(f"pyterrier {pt.__version__}")
print(f"running   {COLLECTION}  ({cfg['topics']} topics, graded={cfg['graded']})")

pyterrier 1.1.2
running   TREC DL'19  (43 topics, graded=True)


## 1b · The index

A prebuilt Terrier index over all 8.8M MS MARCO passages — ~3.2 GB download, ~3.9 GB on disk. Downloaded once, then cached; its own sha256 is verified on the way in.

In [3]:
"""Step 1b — the prebuilt MS MARCO index.

Separate cell because this is the slow, failure-prone step. PyTerrier verifies the artifact's
sha256 as it streams, so a corrupted download raises rather than handing you a subtly broken index.
If that happens, just re-run this cell.
"""
index = pt.Artifact.from_hf("pyterrier/msmarco-passage.terrier")
print(index)

TerrierIndex('/Users/craig.macdonald/.pyterrier/artifacts/5fb9ed05ee653302a044f774b2effcc1e67b60d84a466b82901422fdbea7346c')


## 1c · Topics and qrels

The `/judged` variant only. DL'19 ships 200 topics but just 43 carry relevance judgements — evaluating all 200 averages in zeros and silently divides every metric by ~4.

In [4]:
"""Step 1c — topics and relevance judgements."""
ds     = pt.get_dataset("irds:" + cfg["irds"])
topics = ds.get_topics()      # columns: qid, query
qrels  = ds.get_qrels()       # columns: qid, docno, label

# tripwire: if the wrong variant ever gets loaded, fail here rather than in the final table
assert len(topics) == cfg["topics"], f"expected {cfg['topics']} topics, got {len(topics)}"

print(f"{COLLECTION}: {len(topics)} topics, {len(qrels)} judgements")
print(f"relevance levels present: {sorted(qrels['label'].unique())}")
print(topics.head(3).to_string(index=False))

TREC DL'19: 43 topics, 9260 judgements
relevance levels present: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
    qid                                 query
 156493                      do goldfish grow
1110199             what is wifi vs bluetooth
1063750 why did the us volunterilay enter ww1


## 1d · Retrieve; Only the definition of bm25 is needed

BM25 at the paper's parameters and depth. The `score` column it returns is the raw BM25 value — normally discarded after ranking, but here it is the feature step 2 formats.

In [5]:
"""Step 1d — BM25 retrieval, keeping the raw score."""
# k1/b  depth 1000 
bm25 = index.bm25(k1=BM25_K1, b=BM25_B, num_results=RERANK_DEPTH)

first_stage = bm25.transform(topics)     # columns: qid, docno, score, rank, query

print(f"{len(first_stage):,} candidates, {len(first_stage)/len(topics):.0f} per query")
print(first_stage.head(3)[["qid", "docno", "rank", "score"]].to_string(index=False))

s = first_stage["score"]
print(f"\nraw BM25: min {s.min():.2f} | max {s.max():.2f} | mean {s.mean():.2f} | sd {s.std():.2f}")
print("paper §3.3 reports its own data as {min 0, max 98, mean 7, sd 5}")
print("  -> the MEAN is the number to check. Their DL'19 file gives 6.94, so ours should be close.")
print("  -> the MAX will be lower: 98 came from ~40M training pairs, this is 43 queries.")

Java started (triggered by Retriever.__init__) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


10:23:31.166 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 1.9 GiB of memory would be required.


42,005 candidates, 977 per query
   qid   docno  rank     score
156493 6139386     0 28.013180
156493 8182161     1 27.837165
156493 3288600     2 27.714462

raw BM25: min 7.00 | max 64.99 | mean 18.13 | sd 6.09
paper §3.3 reports its own data as {min 0, max 98, mean 7, sd 5}
  -> the MEAN is the number to check. Their DL'19 file gives 6.94, so ours should be close.
  -> the MAX will be lower: 98 came from ~40M training pairs, this is 43 queries.


## 2a · Normalise the score [ INTERESTING, but only the definition of normalise() is needed ]

`int(raw / 50 * 100)` — Min-Max in the **global** setting, then truncate. The 50 is a fixed constant from §3.3, *not* this query's maximum.

In [6]:
"""Step 2a — turn the raw BM25 float into the integer the model was trained on."""

def normalise(raw: float) -> int:
    return int(((raw - GLOBAL_MIN_BM25) / (GLOBAL_MAX_BM25 - GLOBAL_MIN_BM25)) * 100)

first_stage["inject"] = first_stage["score"].map(normalise)

# worked example, so the arithmetic is checkable by hand
r = first_stage.iloc[0]
print(f"qid {r.qid}  docno {r.docno}")
print(f"  raw BM25        {r.score}")
print(f"  / {GLOBAL_MAX_BM25} * 100   {r.score / GLOBAL_MAX_BM25 * 100}")
print(f"  truncated       {r.inject}")
print(f"  injected as     \"{r.inject}\"  (a string, not a number)")

print(first_stage.head(5)[["qid", "docno", "score", "inject"]].to_string(index=False))

qid 156493  docno 6139386
  raw BM25        28.013180289253768
  / 50 * 100   56.026360578507536
  truncated       56
  injected as     "56"  (a string, not a number)
   qid   docno     score  inject
156493 6139386 28.013180      56
156493 8182161 27.837165      55
156493 3288600 27.714462      55
156493 3288596 27.592916      55
156493 2411918 27.516750      55


## 2b · Why an integer [INTERESTING< but only first 2 lines are necessary]

the integer numbers are already included in the BERT tokenizer's vocabulary, allowing for appropriate tokenization

In [7]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("microsoft/MiniLM-L12-H384-uncased")

multi = [i for i in range(INJECT_MAX_INT + 1)
         if len(tok.encode(str(i), add_special_tokens=False)) != 1]
print(f"integers 0..{INJECT_MAX_INT}: {len(multi)} are not single tokens "
      f"{'-- OK' if not multi else multi[:10]}")

# the alternative the paper rules out, and which a reproduction can easily fall into
for form in ("14", "0.14", "0.14000", "14.4"):
    ids = tok.encode(form, add_special_tokens=False)
    print(f"  {form:9s} -> {len(ids)} token(s)  {tok.convert_ids_to_tokens(ids)}")

integers 0..196: 0 are not single tokens -- OK
  14        -> 1 token(s)  ['14']
  0.14      -> 3 token(s)  ['0', '.', '14']
  0.14000   -> 4 token(s)  ['0', '.', '1400', '##0']
  14.4      -> 3 token(s)  ['14', '.', '4']


## 2c · Do the values land where the model expects? [ INTERESTING, BUT COULD REMOVE]

Training spanned 0–196. If our injected values fall outside that, the model is reading tokens it never saw in this position.

In [8]:
"""Step 2c — check the injected values against the range the model was trained on."""
vals = first_stage["inject"]
over = (vals > INJECT_MAX_INT).sum()

print(f"injected integers: {vals.min()} .. {vals.max()}   mean {vals.mean():.1f}")
print(f"trained range was 0 .. {INJECT_MAX_INT}   -> {over} value(s) above it ({over/len(vals):.2%})")

print("\ndistribution:")
for lo in range(0, 200, 25):
    frac = ((vals >= lo) & (vals < lo + 25)).mean()
    print(f"  {lo:3d}-{lo+24:3d}  {frac:6.1%}  {'#' * int(frac * 50)}")

if over:
    print("\n  WARNING: values above the trained range. Those tokens exist in the vocabulary but")
    print("  the model has never seen them here. The paper does not clip, so neither do we -- record it.")
else:
    print("\n  all inside the trained range")

injected integers: 14 .. 129   mean 35.8
trained range was 0 .. 196   -> 0 value(s) above it (0.00%)

distribution:
    0- 24   10.9%  #####
   25- 49   78.5%  #######################################
   50- 74    9.2%  ####
   75- 99    1.1%  
  100-124    0.2%  
  125-149    0.0%  
  150-174    0.0%  
  175-199    0.0%  

  all inside the trained range


## 3a · Attach the passage text [ ONLY THE FIRST LINE IS NEEDED ]

Retrieval returns document *ids*. The cross-encoder needs the text, and this index stores it

In [9]:
get_text = pt.text.get_text(index, "text")
# we dont run this on thousands of candidates, just a few to sanity check the text is there and the length is reasonable

candidates = get_text(first_stage.head())


print(f"{len(candidates):,} candidates with text")
print(f"passage length: mean {candidates['text'].str.split().str.len().mean():.0f} words "
      f"(paper §4 gives the collection average as 73.1)")
print(f"\nqid    {candidates.iloc[0]['qid']}")
print(f"query  {candidates.iloc[0]['query']}")
print(f"inject {candidates.iloc[0]['inject']}")
print(f"text   {candidates.iloc[0]['text'][:110]}...")

5 candidates with text
passage length: mean 67 words (paper §4 gives the collection average as 73.1)

qid    156493
query  do goldfish grow
inject 56
text   A: The conditions goldfish are kept in plus their diet determine how large they will grow. I have seen goldfis...


## 3b · Build the pair

A cross-encoder tokenises `(text_a, text_b)` as `[CLS] a [SEP] b [SEP]` — no third slot, so the score is spliced into `text_a`. §4 caps the query at 30 tokens and the passage at 200.

In [10]:
def cap(text: str, max_tokens: int) -> str:
    # only touch text that exceeds the cap: decode(encode(x)) is not always x
    ids = tok.encode(text, add_special_tokens=False)
    return text if len(ids) <= max_tokens else tok.decode(ids[:max_tokens], skip_special_tokens=True)

def build_pair(inject, query, passage, score):
    q, p = cap(query, QUERY_MAX_TOKENS), cap(passage, PASSAGE_MAX_TOKENS)
    if inject is None:
        return q, p                          # CE_CAT       [CLS] q [SEP] p [SEP]
    if inject == "score_middle":
        return f"{q} [SEP] {score}", p       # paper Eq.3   [CLS] q [SEP] s [SEP] p [SEP]
    return f"{score} [SEP] {q}", p           # released     [CLS] s [SEP] q [SEP] p [SEP]

# how often does truncation actually fire?
n_q = sum(len(tok.encode(q, add_special_tokens=False)) > QUERY_MAX_TOKENS
          for q in candidates["query"].unique())

          
sample = candidates["text"].sample(min(2000, len(candidates)), random_state=42)
n_p = sum(len(tok.encode(t, add_special_tokens=False)) > PASSAGE_MAX_TOKENS for t in sample)
print(f"queries over {QUERY_MAX_TOKENS} tokens : {n_q} of {candidates['query'].nunique()}")
print(f"passages over {PASSAGE_MAX_TOKENS} tokens: ~{n_p/len(sample):.1%} (of {len(sample)} sampled)")

queries over 30 tokens : 0 of 1
passages over 200 tokens: ~0.0% (of 5 sampled)


## 3c · Check the tokens, don't trust the string [COULD REMOVE]

A literal `[SEP]` in an f-string either resolves to the special token or it doesn't, and nothing downstream would tell you.

In [11]:
Q = "what is a cat"
P = "a cat is a small domesticated carnivorous mammal"
S = 44

print(f"{'arrangement':22s} {'[SEP]':>5s}  tokens")


for label, inject in [("CAT (no injection)", None),
                      ("Eq.3  score_middle", "score_middle"),
                      ("code  score_first", "score_first")]:
    a, b = build_pair(inject, Q, P, S)
    toks = tok.convert_ids_to_tokens(tok(a, b)["input_ids"])
    print(f"{label:22s} {toks.count(tok.sep_token):>5d}  {' '.join(toks[:15])}")

cat = tok.convert_ids_to_tokens(tok(*build_pair(None, Q, P, S))["input_ids"])
mid = tok.convert_ids_to_tokens(tok(*build_pair("score_middle", Q, P, S))["input_ids"])
fst = tok.convert_ids_to_tokens(tok(*build_pair("score_first",  Q, P, S))["input_ids"])



arrangement            [SEP]  tokens
CAT (no injection)         2  [CLS] what is a cat [SEP] a cat is a small domestic ##ated car ##nivorous
Eq.3  score_middle         3  [CLS] what is a cat [SEP] 44 [SEP] a cat is a small domestic ##ated
code  score_first          3  [CLS] 44 [SEP] what is a cat [SEP] a cat is a small domestic ##ated


## 4 · Cross-encoder inference

Score every pair, take the logit. Outputs are raw logits (unbounded, often negative) — only the ordering within a query matters.

In [34]:
BATCH = 128
import torch
def make_cross_encoder(model_id: str, inject: str, device: str = "cuda" if torch.cuda.is_available() else "cpu"):
    from sentence_transformers import CrossEncoder
    model = CrossEncoder(model_id, max_length=256, device=device)
    
    def _score_query(inp : pd.DataFrame) -> pd.DataFrame:
        """Given a DataFrame with columns qid, docno, query, text, inject, score,
        return a DataFrame with columns qid, docno, score (the model's output).
        """
        pt.validate.result_frame(inp, extra_columns=["query", "text", "score"])
        if len(inp) == 0:
            return inp
        
        # the model expects a batch of pairs (query, passage) and returns a batch of scores
        pairs = [ build_pair(inject, row["query"], row["text"], row["score"])  for _, row in inp.iterrows()]
        
        scores = model.predict(pairs, convert_to_numpy=True, batch_size=BATCH).squeeze().tolist()

        return pd.DataFrame({"qid": inp.qid, "docno": inp.docno, "score": scores})
    return pt.apply.by_query(_score_query, add_ranks=True, label='CAT' if inject is None else f'BM25CAT ({inject})')

cross_encoder_cat = make_cross_encoder(MODELS["CAT"]["repo"], MODELS["CAT"]["inject"], device="mps")
cross_encoder_bm25cat = make_cross_encoder(MODELS["BM25CAT"]["repo"], MODELS["BM25CAT"]["inject"], device="mps")


In [35]:
cross_encoder_cat([{"qid" : "Q1", "docno" : "d1", "score": 5, "query": "what are chemical reactions?", "text": "Chemical reactions are processes that lead to the transformation of one or more substances into different substances."}])

[{'qid': 'Q1', 'docno': 'd1', 'score': 0.9605709910392761, 'rank': 0}]

In [36]:
cross_encoder_bm25cat([{"qid" : "Q1", "docno" : "d1", "score": 5, "query": "what are chemical reactions?", "text": "Chemical reactions are processes that lead to the transformation of one or more substances into different substances."}])

[{'qid': 'Q1', 'docno': 'd1', 'score': 0.9372288584709167, 'rank': 0}]

In [37]:
cross_encoder_cat

pt.apply.by_query()

In [38]:
cross_encoder_bm25cat

pt.apply.by_query()

## Formulate pipelines

In [39]:
pipe_cat = bm25 >> get_text >> cross_encoder_cat
pipe_bm25cat = bm25 >> get_text >> pt.apply.doc_score(lambda row: normalise(row["score"]), label="Normalise") >> cross_encoder_bm25cat


Check the whole pipelines work

In [40]:
pipe_cat.search("what are chemical reactions?")

,qid,docno,score,rank
151,1,7575551,0.985732,0
119,1,742207,0.984764,1
630,1,1218080,0.981501,2
862,1,1406142,0.981387,3
762,1,4733213,0.980353,4
...,...,...,...,...
201,1,1710020,0.000473,995
105,1,4323333,0.000469,996
418,1,7470746,0.000452,997
693,1,2880227,0.000439,998


In [41]:
pipe_bm25cat.search("what are chemical reactions?")

,qid,docno,score,rank
762,1,4733213,0.975685,0
672,1,8572191,0.974562,1
95,1,4721960,0.974332,2
379,1,203234,0.974019,3
144,1,2069909,0.973122,4
...,...,...,...,...
492,1,4477078,0.000361,995
749,1,4459954,0.000361,996
535,1,6498972,0.000361,997
872,1,2494175,0.000360,998


## 5c · Evaluate

Two experiments. Everything vs BM25 shows the pipeline works

In [42]:
from IPython.display import display
# we run only the collection selected at the top, but this loop is here to make it easy to run all three in one go.
for ds in [COLLECTION]:
    print(f"\n{ds} results:")
    measures = [pt.measures.nDCG@10, pt.measures.AP(rel=2)@1000, pt.measures.RR(rel=2)@10]
    if not COLLECTIONS[ds]["graded"]:
        measures = [pt.measures.RR@10, pt.measures.NDCG, pt.measures.AP@1000]
    results = pt.Experiment(
        [bm25, pipe_cat, pipe_bm25cat],
        pt.get_dataset("irds:" + COLLECTIONS[ds]["irds"]).get_topics(),
        pt.get_dataset("irds:" + COLLECTIONS[ds]["irds"]).get_qrels(),
        measures,
        baseline=0,
        plan='tree',
        names=["BM25", "CAT", "BM25CAT"]
    )
    print(f"\n{ds} results:")
    display(results)


TREC DL'19 results:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


TREC DL'19 results:


,name,nDCG@10,AP(rel=2)@1000,RR(rel=2)@10,nDCG@10 +,nDCG@10 -,nDCG@10 p-value,AP(rel=2)@1000 +,AP(rel=2)@1000 -,AP(rel=2)@1000 p-value,RR(rel=2)@10 +,RR(rel=2)@10 -,RR(rel=2)@10 p-value
0,BM25,0.504820,0.299747,0.672185,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CAT,0.689491,0.456544,0.825581,35.0,8.0,0.000011,36.0,6.0,0.000001,15.0,4.0,0.035458
2,BM25CAT,0.661734,0.429713,0.820543,33.0,10.0,0.000069,34.0,8.0,0.000038,17.0,5.0,0.044976
